In [47]:
# --- Make project imports work no matter where the notebook runs from ---
from pathlib import Path
import sys
import os

def find_project_root():
    here = Path.cwd()
    # Prefer a directory that has both 'utils' and 'models'
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    # Fallback: look for common markers (git, requirements, pyproject)
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here  # last resort

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root set to:", PROJECT_ROOT)

# Optional: nice for iterative development
%load_ext autoreload
%autoreload 2


import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
import contextlib
import plotly.graph_objects as go
from torch.cuda.amp import autocast, GradScaler

from utils.reproducibility_utils import seed_everything
from utils.data_loadingV2 import load_data, load_test_data, feature_names
from utils.WeightedL1Loss import WeightedHybridLoss
from utils.LDS_FocalR_Loss import EnhancedLDS_FocalR_Loss
from utils.kaggle_metric import score

from models.lstm import LSTMModel
from models.gru import GRUModel
from models.Transformer_ImbalancedRegression import TransformerRegressor

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')


Project root set to: /home/mdesouky/workspace/mait_g2_drift_challenge
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
CONFIG = {
    # PATHS
    'TRAIN_PATH': "../datasets_challenge/train",
    'TEST_PATH': "../datasets_challenge/test",
    'DIST_CSV': "../datasets_challenge/sideSlip_distribution.csv",
    'CHECKPOINT_NAME': "best_modelv5.pth",
    'SUBMISSION_CSV': "predictions.csv",
    
    # REPRODUCIBILITY
    'SEED': 42,
    
    # DATA LOADING
    'TRAIN_RATIO': 0.8,
    'BATCH_SIZE': 512,
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': "vxCG",
    'COLUMNS_TO_DROP': ["sideSlip", "vyCG", "time", "longitude", "latitude"],
    'TARGET': "sideSlip",
    'SCALING': True,
    
    # MODEL ARCHITECTURE 
    'MODEL_TYPE': 'LSTM',  # 🔄 BACK TO LSTM - proven to work (iter 12: 0.208°)
    'RNN_UNITS': 256,          
    'FC_UNITS': 256,        
    'NUM_LAYERS': 4,           
    'DROPOUT': [0.18, 0.12, 0.08],  # ✅ Proven dropout from iter 13b
    'USE_BATCH_NORM': True,
    
    # LOSS CONFIGURATION (BALANCED AGGRESSIVE - Option A)
    'LOSS_TYPE': 'LDS_FocalR',  # 'WeightedHybrid' or 'LDS_FocalR'
    'LOSS_THRESHOLDS': (0.1, 0.4),
    'LOSS_WEIGHTS': (4.0, 2.0, 3.5),  # 🎯 TUNED: Many=4.0 (pass constraint), Few=3.5 (hit target)
    'USE_HUBER': True,
    'HUBER_BETA': 0.08,  # ✅ Proven optimal value
    
    # LDS & FOCAL PARAMETERS (INCREASED - Stronger hard sample focus)
    'LDS_KERNEL': 'gaussian',
    'LDS_KS': 7,  # ✅ Proven optimal kernel size
    'LDS_SIGMA': 1.8,  # 🎯 TUNED: 1.5→1.8 (more tail smoothing)
    'FOCAL_GAMMA': 0.6,  # 🎯 TUNED: 0.5→0.6 (stronger hard sample focus)
    'NUM_BINS': 100,
    
    # OPTIMIZER (WITH COSINE ANNEALING FOR FINE-TUNING)
    'LEARNING_RATE': 7e-4,  # 🔄 LSTM proven optimal range
    'WEIGHT_DECAY': 5e-4,  # ✅ Keep balanced regularization
    'BETAS': (0.9, 0.999),
    'MAX_GRAD_NORM': 0.7,  # 🔄 LSTM proven value
    
    # SCHEDULER (ENABLED - needed to escape plateau!)
    'USE_SCHEDULER': True,  # 🔄 RE-ENABLED: Fixed LR caused plateau at 0.26°
    'SCHEDULER_T_MAX': 300,  # Decay over 300 epochs
    'SCHEDULER_ETA_MIN': 1e-6,  # Minimum LR
    
    # TRAINING
    'NUM_EPOCHS': 500,  # 🎯 TUNED: 400→500 (more time to converge)
    'EARLY_STOP_PATIENCE': 120,  # 🎯 TUNED: 100→120 (more patience)
    'USE_AMP': True,
    'EMA_DECAY': 0.995,  # ✅ Proven stable value
    
    # IMBALANCED SAMPLING (NEW - Help few-shot learning)
    'USE_IMBALANCED_SAMPLER': False,  # 🔄 DISABLED - causing instability in early epochs
    'SAMPLER_BUCKET_PROBS': (0.35, 0.30, 0.35),  # 🎯 Equal focus: Many/Medium/Few
    'SAMPLER_OVERSAMPLING': 1.5,  # 1.5x dataset size per epoch
    
    # AUGMENTATION (NEW - Regularization for few-shot)
    'USE_MIXUP': False,  # 🔄 Disable for now (test baseline first)
    'MIXUP_ALPHA': 0.2,
    'MIXUP_PROB': 0.3,        
    
    # CHALLENGE CONSTRAINTS
    'MANY_SHOT_CONSTRAINT': 0.2,
    'FEW_SHOT_TARGET': 0.3,
    
    # VISUALIZATION
    'FIG_SIZE': (14, 8),
    'PLOTLY_WIDTH': 1200,
    'PLOTLY_HEIGHT': 500,
}


🚀 CONFIGURATION LOADED - BALANCED AGGRESSIVE (OPTION A)
🎯 STRATEGY: Hit BOTH Targets with Tuned Weights
   Constraint (Hard): Many-shot MAE ≤ 0.20°
   Objective (Primary): Few-shot MAE < 0.30°
   Tuning: Increased weights (4.0, 2.0, 3.5) + focal=0.6 + sigma=1.8
   Based on: V2 results (Many=0.2221°, Few=0.6018°)
🔄 LSTM ARCHITECTURE (PROVEN):
   Model: LSTM
   RNN Units: 256
   FC Units: 256
   Layers: 4
   Dropout: [0.18, 0.12, 0.08]
   Batch Norm: True
   Parameters: ~400K (optimal for 20-step sequences)
   Evidence: Iter 12 achieved 0.208° (closest to constraint)
⚙️  TRAINING CONFIGURATION:
   Learning Rate: 7.0e-04 → 1.0e-06 (Cosine decay)
   Scheduler: ENABLED ✅ (T_max=300)
   Weight Decay: 5.0e-04 | Grad Clip: 0.7
   Batch Size: 512 | Epochs: 500
   EMA Decay: 0.995 (exponential moving average)
   Early Stop: 120 epochs patience
🎲 LOSS FUNCTION: EnhancedLDS_FocalR_Loss
   Bucket Weights: (4.0, 2.0, 3.5) (Many, Medium, Few)
   Strategy: BALANCED AGGRESSIVE - Hit both targets
   Rat

In [49]:
# filepath: notebooks/task2v3.ipynb
seed_everything(CONFIG['SEED'])
print(f"✓ Random seed set to {CONFIG['SEED']}")

✓ Random seed set to 42


In [ ]:

train_dataloader, val_dataloader, pipeline = load_data(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=CONFIG['TRAIN_RATIO'],
    batchsize=CONFIG['BATCH_SIZE'],
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    scaling=CONFIG['SCALING'],
    seed=CONFIG['SEED'],
    verbose=True,  # Show split details
)

print(f"\n✓ Training batches: {len(train_dataloader)}")
print(f"✓ Validation batches: {len(val_dataloader)}")


Loading training data from: ../datasets_challenge/train
🔄 USING data_loadingV2.py - Challenge-Accurate Temporal Split
   Each CSV split individually to prevent temporal leakage
train_part_001.csv: Train=9473, Val=2368, Block=[3392:5760]
train_part_003.csv: Train=7895, Val=1973, Block=[7115:9088]
train_part_004.csv: Train=1145, Val=285, Block=[837:1122]
train_part_005.csv: Train=1221, Val=304, Block=[1159:1463]
train_part_006.csv: Train=16405, Val=4101, Block=[15872:19973]
train_part_007.csv: Train=1228, Val=306, Block=[426:732]
train_part_008.csv: Train=17777, Val=4443, Block=[14000:18443]
train_part_009.csv: Train=6176, Val=1543, Block=[2105:3648]
train_part_010.csv: Train=7567, Val=1891, Block=[7157:9048]
train_part_011.csv: Train=12416, Val=3103, Block=[7705:10808]
train_part_012.csv: Train=897, Val=224, Block=[111:335]
train_part_013.csv: Train=7866, Val=1966, Block=[4557:6523]
train_part_015.csv: Train=6678, Val=1669, Block=[3023:4692]
train_part_008.csv: Train=17777, Val=4443, B

In [58]:
# ============================================================================
# MODEL & LOSS SETUP
# ============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# AMP setup
if torch.cuda.is_available():
    amp_device_type = "cuda"
    prefer_bf16 = True
    amp_dtype = torch.bfloat16 if (prefer_bf16 and torch.cuda.is_bf16_supported()) else torch.float16
    scaler = torch.amp.GradScaler(amp_device_type, enabled=CONFIG['USE_AMP'])
else:
    amp_device_type = None
    amp_dtype = None
    scaler = torch.amp.GradScaler("cuda", enabled=False)

def autocast_ctx():
    if amp_device_type is None or not CONFIG['USE_AMP']:
        return contextlib.nullcontext()
    return torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=True)

# Get input size from data
X_sample, _ = next(iter(train_dataloader))
input_size = X_sample.shape[2]
seq_length = X_sample.shape[1]

# Create model based on MODEL_TYPE
if CONFIG['MODEL_TYPE'] == 'LSTM':
    model = LSTMModel(
        input_size=input_size,
        output_size=1,
        rnn_units=CONFIG['RNN_UNITS'],
        fc_units=CONFIG['FC_UNITS'],
        dropout=CONFIG['DROPOUT'],
        num_layers=CONFIG['NUM_LAYERS'],
        use_batch_norm=CONFIG['USE_BATCH_NORM']
    ).to(device)
elif CONFIG['MODEL_TYPE'] == 'GRU':
    model = GRUModel(
        input_size=input_size,
        output_size=1,
        rnn_units=CONFIG['RNN_UNITS'],
        fc_units=CONFIG['FC_UNITS'],
        dropout=CONFIG['DROPOUT'],
        num_layers=CONFIG['NUM_LAYERS'],
        use_batch_norm=CONFIG['USE_BATCH_NORM']
    ).to(device)
elif CONFIG['MODEL_TYPE'] == 'Transformer':
    model = TransformerRegressor(
        input_size=input_size,
        d_model=CONFIG['TRANSFORMER_D_MODEL'],
        nhead=CONFIG['TRANSFORMER_NHEAD'],
        num_layers=CONFIG['TRANSFORMER_NUM_LAYERS'],
        dim_feedforward=CONFIG['TRANSFORMER_DIM_FFN'],
        dropout=CONFIG['TRANSFORMER_DROPOUT'],
        max_seq_len=max(seq_length, CONFIG['TRANSFORMER_MAX_SEQ_LEN'])
    ).to(device)
else:
    raise ValueError(f"Unknown MODEL_TYPE: {CONFIG['MODEL_TYPE']}")

# Create loss function
if CONFIG['LOSS_TYPE'] == 'WeightedHybrid':
    criterion = WeightedHybridLoss(
        thresholds=CONFIG['LOSS_THRESHOLDS'],
        weights=CONFIG['LOSS_WEIGHTS'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        scaler_y=None,
        target_is_scaled=False
    )
    print(f"✓ Loss: WeightedHybridLoss")
    print(f"  Thresholds: {CONFIG['LOSS_THRESHOLDS']}")
    print(f"  Weights: {CONFIG['LOSS_WEIGHTS']}")
    print(f"  Huber: {CONFIG['USE_HUBER']} (beta={CONFIG['HUBER_BETA']})")
else:
    criterion = EnhancedLDS_FocalR_Loss(
        train_targets=y_train_np,
        bucket_weights=CONFIG['LOSS_WEIGHTS'],
        lds_kernel=CONFIG['LDS_KERNEL'],
        lds_ks=CONFIG['LDS_KS'],
        lds_sigma=CONFIG['LDS_SIGMA'],
        focal_gamma=CONFIG['FOCAL_GAMMA'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        num_bins=CONFIG['NUM_BINS'],
        dist_csv_path=CONFIG['DIST_CSV']
    ).to(device)
    print(f"✓ Loss: EnhancedLDS_FocalR_Loss")
    print(f"  Bucket Weights: {CONFIG['LOSS_WEIGHTS']} (Many, Medium, Few)")
    print(f"  LDS: kernel={CONFIG['LDS_KERNEL']}, ks={CONFIG['LDS_KS']}, sigma={CONFIG['LDS_SIGMA']}")
    print(f"  Focal: gamma={CONFIG['FOCAL_GAMMA']}")
    print(f"  Huber: {CONFIG['USE_HUBER']} (beta={CONFIG['HUBER_BETA']})")
    print(f"  Bins: {CONFIG['NUM_BINS']}")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*70}")
print(f"MODEL: {CONFIG['MODEL_TYPE']}")
print(f"{'='*70}")
print(f"Parameters: {total_params:,} ({trainable_params:,} trainable)")
print(f"Size: {total_params * 4 / 1024**2:.2f} MB")
print(f"Architecture:")
print(f"  - Input Size: {input_size}")
print(f"  - Sequence Length: {seq_length}")
if CONFIG['MODEL_TYPE'] in ['LSTM', 'GRU']:
    print(f"  - RNN Units: {CONFIG['RNN_UNITS']}")
    print(f"  - FC Units: {CONFIG['FC_UNITS']}")
    print(f"  - Layers: {CONFIG['NUM_LAYERS']}")
    print(f"  - Dropout: {CONFIG['DROPOUT']}")
    print(f"  - Batch Norm: {CONFIG['USE_BATCH_NORM']}")
else:  # Transformer
    print(f"  - d_model: {CONFIG['TRANSFORMER_D_MODEL']}")
    print(f"  - nhead: {CONFIG['TRANSFORMER_NHEAD']}")
    print(f"  - num_layers: {CONFIG['TRANSFORMER_NUM_LAYERS']}")
    print(f"  - dim_feedforward: {CONFIG['TRANSFORMER_DIM_FFN']}")
    print(f"  - dropout: {CONFIG['TRANSFORMER_DROPOUT']}")
    print(f"  - max_seq_len: {max(seq_length, CONFIG['TRANSFORMER_MAX_SEQ_LEN'])}")
print(f"{'='*70}\n")


DEVICE: cuda
GPU: NVIDIA RTX A4000


[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:    99538 (90.82%)
  Medium:   7613 ( 6.95%)
  Few:      2448 ( 2.23%)
Bucket weights: 4.0 / 2.0 / 3.5
Focal gamma: 0.6 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=1.8

✓ Loss: EnhancedLDS_FocalR_Loss
  Bucket Weights: (4.0, 2.0, 3.5) (Many, Medium, Few)
  LDS: kernel=gaussian, ks=7, sigma=1.8
  Focal: gamma=0.6
  Huber: True (beta=0.08)
  Bins: 100

MODEL: LSTM
Parameters: 1,965,101 (1,965,101 trainable)
Size: 7.50 MB
Architecture:
  - Input Size: 22
  - Sequence Length: 20
  - RNN Units: 256
  - FC Units: 256
  - Layers: 4
  - Dropout: [0.18, 0.12, 0.08]
  - Batch Norm: True


[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):


In [23]:
# ============================================================================
# TRAINING SETUP (Adapted from task_01v6)
# ============================================================================

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    betas=CONFIG['BETAS']
)

# Conditional scheduler setup
if CONFIG.get('USE_SCHEDULER', True):
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG['SCHEDULER_T_MAX'],
        eta_min=CONFIG['SCHEDULER_ETA_MIN']
    )
    print(f"✓ Optimizer: AdamW (lr={CONFIG['LEARNING_RATE']:.1e}, wd={CONFIG['WEIGHT_DECAY']:.1e})")
    print(f"✓ Scheduler: CosineAnnealingLR (T_max={CONFIG['SCHEDULER_T_MAX']}, eta_min={CONFIG['SCHEDULER_ETA_MIN']:.1e})")
else:
    scheduler = None
    print(f"✓ Optimizer: AdamW (lr={CONFIG['LEARNING_RATE']:.1e}, wd={CONFIG['WEIGHT_DECAY']:.1e})")
    print(f"✅ Scheduler: DISABLED - Using FIXED learning rate {CONFIG['LEARNING_RATE']:.1e}")

# ✅ NEW: EMA (Exponential Moving Average) from task_01v6
ema_params = [p.detach().clone() for p in model.parameters()]

def ema_update(model_params, ema_params, decay):
    """Update EMA parameters"""
    for p, e in zip(model_params, ema_params):
        e.mul_(decay).add_(p.detach(), alpha=1.0 - decay)

@torch.no_grad()
def load_params_(model_params, src_params):
    """Load parameters from source"""
    for p, s in zip(model_params, src_params):
        p.copy_(s)

print(f"✓ EMA: Enabled (decay={CONFIG['EMA_DECAY']})")
print(f"✓ Gradient Clipping: max_norm={CONFIG['MAX_GRAD_NORM']}")

# Training state
train_curve, val_curve = [], []
train_mae_curve, val_mae_curve = [], []
bucket_many, bucket_medium, bucket_few = [], [], []
learning_rate_curve = []

best_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0

print(f"\n🎯 TRAINING STRATEGY:")
print(f"   Evaluate BOTH live weights and EMA")
print(f"   Save best performer (live or EMA)")
print(f"   Constraint: Many-shot MAE ≤ {CONFIG['MANY_SHOT_CONSTRAINT']:.2f}°")
print(f"   Early stop after {CONFIG['EARLY_STOP_PATIENCE']} epochs without improvement\n")

✓ Optimizer: AdamW (lr=7.0e-04, wd=5.0e-04)
✓ Scheduler: CosineAnnealingLR (T_max=300, eta_min=1.0e-06)
✓ EMA: Enabled (decay=0.995)
✓ Gradient Clipping: max_norm=0.7

🎯 TRAINING STRATEGY:
   Evaluate BOTH live weights and EMA
   Save best performer (live or EMA)
   Constraint: Many-shot MAE ≤ 0.20°
   Early stop after 120 epochs without improvement



In [24]:
# ============================================================================
# TRAINING LOOP (Adapted from task_01v6 with EMA and dual evaluation)
# ============================================================================

for epoch in range(CONFIG['NUM_EPOCHS']):
    # =============== TRAINING ===============
    model.train()
    running_loss, running_mae = 0.0, 0.0
    num_train_batches = len(train_dataloader)

    for step, (xb, yb) in enumerate(train_dataloader):
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float().reshape(-1, 1)

        optimizer.zero_grad(set_to_none=True)
        
        # Forward pass with AMP
        with autocast_ctx():
            pred = model(xb)
            total_loss = criterion(pred, yb)
        
        # Backward pass with gradient clipping
        scaler.scale(total_loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['MAX_GRAD_NORM'])
        scaler.step(optimizer)
        scaler.update()

        # ✅ EMA update per batch
        ema_update(model.parameters(), ema_params, CONFIG['EMA_DECAY'])

        running_loss += total_loss.item()
        running_mae += (pred - yb).abs().mean().item()

    avg_train_loss = running_loss / num_train_batches
    avg_train_mae = running_mae / num_train_batches
    train_curve.append(avg_train_loss)
    train_mae_curve.append(avg_train_mae)

    # =============== VALIDATION (Dual Evaluation: Live vs EMA) ===============
    model.eval()
    
    def _evaluate(current_params):
        """Evaluate with specific parameters"""
        snapshot = [p.detach().clone() for p in model.parameters()]
        load_params_(model.parameters(), current_params)
        
        running_vloss, running_vmae = 0.0, 0.0
        all_val_preds, all_val_targets = [], []
        num_val_batches = len(val_dataloader)
        
        with torch.no_grad():
            for xv, yv in val_dataloader:
                xv = xv.to(device, non_blocking=True).float()
                yv = yv.to(device, non_blocking=True).float().reshape(-1, 1)
                
                with autocast_ctx():
                    pred_val = model(xv)
                    total_loss_val = criterion(pred_val, yv)
                
                running_vloss += total_loss_val.item()
                running_vmae += (pred_val - yv).abs().mean().item()
                all_val_preds.append(pred_val.cpu())
                all_val_targets.append(yv.cpu())
        
        # Compute bucket metrics
        all_val_preds = torch.cat(all_val_preds, dim=0).to(device)
        all_val_targets = torch.cat(all_val_targets, dim=0).to(device)
        
        if CONFIG['LOSS_TYPE'] == 'WeightedHybrid':
            bucket_stats = criterion.bucket_mae(all_val_preds.squeeze(-1), all_val_targets.squeeze(-1))
        else:
            bucket_stats = criterion.bucket_metrics(all_val_preds, all_val_targets)
        
        avg_vloss = running_vloss / num_val_batches
        avg_vmae = running_vmae / num_val_batches
        
        # Restore original params
        load_params_(model.parameters(), snapshot)
        return avg_vloss, avg_vmae, bucket_stats
    
    # Evaluate both live and EMA weights
    live_params = [p.detach().clone() for p in model.parameters()]
    vloss_live, vmae_live, buckets_live = _evaluate(live_params)
    vloss_ema, vmae_ema, buckets_ema = _evaluate(ema_params)
    
    # Choose better performer
    if vloss_ema <= vloss_live:
        best_view = "EMA"
        best_vloss, best_vmae = vloss_ema, vmae_ema
        best_buckets = buckets_ema
        best_params = [p.detach().clone() for p in ema_params]
    else:
        best_view = "LIVE"
        best_vloss, best_vmae = vloss_live, vmae_live
        best_buckets = buckets_live
        best_params = live_params
    
    val_curve.append(best_vloss)
    val_mae_curve.append(best_vmae)
    
    # Extract bucket MAE
    if CONFIG['LOSS_TYPE'] == 'WeightedHybrid':
        many_mae = best_buckets['mae_b0']
        med_mae = best_buckets['mae_b1']
        few_mae = best_buckets['mae_b2']
    else:
        many_mae = best_buckets['mae_many']
        med_mae = best_buckets['mae_medium']
        few_mae = best_buckets['mae_few']
    
    bucket_many.append(many_mae)
    bucket_medium.append(med_mae)
    bucket_few.append(few_mae)
    
    # Step scheduler (if enabled)
    current_lr = optimizer.param_groups[0]['lr']
    learning_rate_curve.append(current_lr)
    if scheduler is not None:
        scheduler.step()

    # =============== LOGGING ===============
    constraint_symbol = "🎯" if many_mae <= CONFIG['MANY_SHOT_CONSTRAINT'] else ""
    print(f"Epoch {epoch+1:3d}/{CONFIG['NUM_EPOCHS']} | "
          f"TrL: {avg_train_loss:.5f} | VaL: {best_vloss:.5f} | "
          f"TrMAE: {avg_train_mae:.5f} | VaMAE: {best_vmae:.5f} | "
          f"Many: {many_mae:.4f}{constraint_symbol} | Med: {med_mae:.4f} | Few: {few_mae:.4f} | "
          f"LR: {current_lr:.6f} | {best_view}", end="")

    # =============== CHECKPOINTING (Constraint-Aware) ===============
    # Prioritize models that pass constraint, then minimize validation loss
    passes_constraint = many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']
    
    # Determine if this is better than current best
    if best_epoch == -1:
        # First epoch always saves
        improved = True
    elif passes_constraint:
        # If current passes constraint, save if:
        # 1. Previous didn't pass, OR
        # 2. Both pass but this has lower validation loss
        prev_passes = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
        improved = (not prev_passes) or (prev_passes and best_vloss < best_loss - 1e-9)
    else:
        # If current doesn't pass, only save if previous also doesn't pass AND loss improved
        prev_passes = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
        improved = (not prev_passes) and (best_vloss < best_loss - 1e-9)
    
    if improved:
        best_loss = best_vloss
        best_epoch = epoch
        epochs_no_improve = 0
        
        checkpoint_dict = {
            'epoch': epoch,
            'model_type': CONFIG['MODEL_TYPE'],
            'model_state_dict': model.state_dict(),
            'best_params': best_params,
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': best_vloss,
            'train_mae': avg_train_mae,
            'val_mae': best_vmae,
            'bucket_stats': best_buckets,
            'many_mae': many_mae,
            'few_mae': few_mae,
            'best_view': best_view,
            'config': CONFIG
        }
        if scheduler is not None:
            checkpoint_dict['scheduler_state_dict'] = scheduler.state_dict()
        torch.save(checkpoint_dict, CONFIG['CHECKPOINT_NAME'])
        
        # Show why this was saved
        if passes_constraint:
            print(f" ✅ BEST ({best_view}) - PASSES CONSTRAINT!")
        else:
            print(f" ✅ BEST ({best_view})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve > CONFIG['EARLY_STOP_PATIENCE']:
            print(f"\n\n{'='*80}")
            print(f"⏹ Early stopping at epoch {epoch+1} (best: epoch {best_epoch+1})")
            print(f"{'='*80}")
            break
        else:
            print("")

print(f"\n{'='*80}")
print(f"✓ TRAINING COMPLETE")
print(f"  Best Epoch: {best_epoch+1}/{len(train_curve)}")
print(f"  Best Val Loss: {best_loss:.5f}")
if best_epoch >= 0:
    print(f"  Bucket MAE: Many={bucket_many[best_epoch]:.5f} | "
          f"Med={bucket_medium[best_epoch]:.5f} | Few={bucket_few[best_epoch]:.5f}")
print(f"{'='*80}\n")

# =============== LOAD BEST CHECKPOINT ===============
checkpoint = torch.load(CONFIG['CHECKPOINT_NAME'], map_location=device)

# Load the best params (EMA or live)
if 'best_params' in checkpoint:
    state_dict = model.state_dict()
    named_params = [n for n, _ in model.named_parameters()]
    restored = {name: tensor.to(device) for name, tensor in zip(named_params, checkpoint['best_params'])}
    state_dict.update(restored)
    model.load_state_dict(state_dict)
else:
    model.load_state_dict(checkpoint['model_state_dict'])

model.to(device).eval()

print(f"✓ Best model loaded: Epoch {checkpoint['epoch']+1}")
print(f"  View: {checkpoint.get('best_view', 'N/A')}")
print(f"  Val Loss: {checkpoint['val_loss']:.5f} | Val MAE: {checkpoint['val_mae']:.5f}")
print(f"  Many: {checkpoint['many_mae']:.4f}° | Few: {checkpoint['few_mae']:.4f}°\n")

Epoch   1/500 | TrL: 2.07125 | VaL: 1.00083 | TrMAE: 1.05215 | VaMAE: 0.75926 | Many: 0.7295 | Med: 0.8960 | Few: 1.6581 | LR: 0.000700 | LIVE ✅ BEST (LIVE)
Epoch   2/500 | TrL: 1.60855 | VaL: 0.90921 | TrMAE: 0.89640 | VaMAE: 0.66493 | Many: 0.6366 | Med: 0.7921 | Few: 1.4761 | LR: 0.000700 | EMA ✅ BEST (EMA)
Epoch   2/500 | TrL: 1.60855 | VaL: 0.90921 | TrMAE: 0.89640 | VaMAE: 0.66493 | Many: 0.6366 | Med: 0.7921 | Few: 1.4761 | LR: 0.000700 | EMA ✅ BEST (EMA)
Epoch   3/500 | TrL: 1.46219 | VaL: 0.92903 | TrMAE: 0.76728 | VaMAE: 0.74319 | Many: 0.7322 | Med: 0.7484 | Few: 1.2825 | LR: 0.000700 | LIVE
Epoch   3/500 | TrL: 1.46219 | VaL: 0.92903 | TrMAE: 0.76728 | VaMAE: 0.74319 | Many: 0.7322 | Med: 0.7484 | Few: 1.2825 | LR: 0.000700 | LIVE
Epoch   4/500 | TrL: 1.47183 | VaL: 0.87959 | TrMAE: 0.77059 | VaMAE: 0.61116 | Many: 0.5794 | Med: 0.6850 | Few: 1.7724 | LR: 0.000700 | LIVE ✅ BEST (LIVE)
Epoch   4/500 | TrL: 1.47183 | VaL: 0.87959 | TrMAE: 0.77059 | VaMAE: 0.61116 | Many: 0.57

KeyboardInterrupt: 

In [26]:
checkpoint = torch.load(CONFIG['CHECKPOINT_NAME'], map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()

print(f"✓ Loaded checkpoint: {CONFIG['CHECKPOINT_NAME']}")
print(f"  Epoch: {checkpoint['epoch']+1}")
print(f"  Val Loss: {checkpoint['val_loss']:.5f}")
print(f"  Val MAE: {checkpoint['val_mae']:.5f}")

# Handle different bucket stat keys based on loss type
bucket_stats = checkpoint['bucket_stats']
if 'mae_many' in bucket_stats:
    print(f"  Many:   {bucket_stats['mae_many']:.4f}°")
    print(f"  Medium: {bucket_stats['mae_medium']:.4f}°")
    print(f"  Few:    {bucket_stats['mae_few']:.4f}°")
else:
    print(f"  Many:   {bucket_stats['mae_b0']:.4f}°")
    print(f"  Medium: {bucket_stats['mae_b1']:.4f}°")
    print(f"  Few:    {bucket_stats['mae_b2']:.4f}°")

✓ Loaded checkpoint: best_modelv5.pth
  Epoch: 187
  Val Loss: 0.34571
  Val MAE: 0.24453
  Many:   0.2231°
  Medium: 0.4651°
  Few:    0.5972°


In [27]:
# ============================================================================
# VALIDATION VERIFICATION
# ============================================================================

model.eval()
running_val_loss, running_val_mae = 0.0, 0.0

with torch.no_grad():
    for xv, yv in val_dataloader:
        xv = xv.to(device).float()
        yv = yv.to(device).float().reshape(-1, 1)
        pred_val = model(xv)
        val_loss = criterion(pred_val, yv)
        running_val_loss += val_loss.item()
        running_val_mae += (pred_val - yv).abs().mean().item()

avg_val_loss = running_val_loss / len(val_dataloader)
avg_val_mae = running_val_mae / len(val_dataloader)

print(f"{'='*80}")
print(f"VALIDATION VERIFICATION")
print(f"{'='*80}")
print(f"Val Loss: {avg_val_loss:.5f}")
print(f"Val MAE:  {avg_val_mae:.5f}°")
print(f"{'='*80}\n")

VALIDATION VERIFICATION
Val Loss: 0.34566
Val MAE:  0.24460°



In [28]:
# ============================================================================
# VISUALIZE PREDICTIONS ON TRAINING DATA
# ============================================================================

print("Loading complete training data (unshuffled) for visualization...")
train_dataloader_unshuffled = load_test_data(
    test_path=CONFIG['TRAIN_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    load_anonymized_data=False  # We have ground truth
)

model.eval()
total_loss = 0
preds = []
ground_truths = []

with torch.no_grad():
    for step, (x, y) in enumerate(train_dataloader_unshuffled):
        x, y = x.to(device), y.to(device).reshape(-1, 1)
        pred = model(x)
        loss = criterion(pred, y)
        total_loss += loss.item()
        
        preds.append(pred.cpu().numpy())
        ground_truths.append(y.cpu().numpy())

avg_loss = total_loss / len(train_dataloader_unshuffled)
ground_truths = np.concatenate(ground_truths).flatten()
preds = np.concatenate(preds).flatten()

print(f"Avg Loss on Training Data: {avg_loss:.5f}")
print(f"MAE: {np.mean(np.abs(ground_truths - preds)):.5f}°")
print(f"Predictions shape: {preds.shape}\n")

# Interactive Plotly visualization
fig = go.Figure()
fig.add_trace(go.Scatter(y=ground_truths, mode='lines', name='Ground Truth', 
                         line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(y=preds, mode='lines', name='Predictions', 
                         line=dict(color='red', width=1), opacity=0.7))
fig.update_layout(
    title='Training Data: Ground Truth vs Predictions',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (°)',
    width=CONFIG['PLOTLY_WIDTH'],
    height=CONFIG['PLOTLY_HEIGHT']
)
fig.show()

Loading complete training data (unshuffled) for visualization...
Avg Loss on Training Data: 1.66628
MAE: 0.72506°
Predictions shape: (274786,)

Avg Loss on Training Data: 1.66628
MAE: 0.72506°
Predictions shape: (274786,)



In [29]:
# ============================================================================
# EVALUATE WITH CHALLENGE METRIC
# ============================================================================

print("Evaluating with official challenge metric...\n")

# Load distribution for band mapping
dist_df = pd.read_csv(CONFIG['DIST_CSV'])
shot_band_mapping = dist_df.set_index("sideSlip")["band"].to_dict()

# Create ground truth dataframe
gt_df = pd.DataFrame({"ground_truth_sideslip": ground_truths}, dtype=np.float64)
gt_df["id"] = range(len(ground_truths))
gt_df["band"] = gt_df["ground_truth_sideslip"].round(1).map(shot_band_mapping).fillna("unknown")

# Create predictions dataframe
preds_df = pd.DataFrame({"id": range(len(preds)), "predicted_sideslip": preds})

# Calculate score
score_result = score(gt_df, preds_df, row_id_column_name="id")

print(f"\n{'='*80}")
print(f"CHALLENGE METRIC RESULTS")
print(f"{'='*80}")
print(f"Band Distribution:")
print(gt_df["band"].value_counts())
print(f"{'='*80}\n")

Evaluating with official challenge metric...



ParticipantVisibleError: Your mean absolute error for the many shot region is 0.623, which is above the threshold of 0.2. Please improve your model for these shots.

In [ ]:
# ============================================================================
# TRAINING VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=CONFIG['FIG_SIZE'])

# Loss curves
axes[0, 0].plot(train_curve, label="Train", color='blue', linewidth=2)
axes[0, 0].plot(val_curve, label="Validation", color='orange', linewidth=2)
axes[0, 0].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE curves
axes[0, 1].plot(train_mae_curve, label="Train MAE", color='green', linewidth=2)
axes[0, 1].plot(val_mae_curve, label="Val MAE", color='red', linewidth=2)
axes[0, 1].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("MAE (°)")
axes[0, 1].set_title("Mean Absolute Error")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bucket MAE evolution
axes[1, 0].plot(bucket_many, label="Many-shot", color='blue', linewidth=2)
axes[1, 0].plot(bucket_medium, label="Medium-shot", color='orange', linewidth=2)
axes[1, 0].plot(bucket_few, label="Few-shot", color='red', linewidth=2)
axes[1, 0].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2)
axes[1, 0].axhline(y=CONFIG['MANY_SHOT_CONSTRAINT'], color='black', linestyle=':', 
                   linewidth=2, label=f"Constraint ({CONFIG['MANY_SHOT_CONSTRAINT']}°)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Bucket MAE (°)")
axes[1, 0].set_title("Bucket-wise Performance")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning rate schedule
axes[1, 1].plot(learning_rate_curve, color='purple', linewidth=2, marker='o', markersize=3)
axes[1, 1].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].set_title("Learning Rate Schedule")
axes[1, 1].set_yscale('log')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Training summary
print(f"\n{'='*80}")
print(f"TRAINING SUMMARY - {CONFIG['MODEL_TYPE']}")
print(f"{'='*80}")
print(f"Best Epoch: {best_epoch + 1}/{len(train_curve)}")
print(f"Best Val Loss: {min(val_curve):.5f}")
print(f"Best Val MAE: {min(val_mae_curve):.5f}")
print(f"Final LR: {learning_rate_curve[-1]:.6f}")
print(f"Early Stop: {'Yes' if len(train_curve) < CONFIG['NUM_EPOCHS'] else 'No'}")
print(f"\nBucket MAE at Best Epoch:")
print(f"  Many-shot:   {bucket_many[best_epoch]:.4f}°")
print(f"  Medium-shot: {bucket_medium[best_epoch]:.4f}°")
print(f"  Few-shot:    {bucket_few[best_epoch]:.4f}°")
print(f"{'='*80}\n")

In [ ]:
# ============================================================================
# GENERATE PREDICTIONS
# ============================================================================

print(f"\nLoading test data from: {CONFIG['TEST_PATH']}")
test_dataloader = load_test_data(
    test_path=CONFIG['TEST_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    load_anonymized_data=True,
)

print(f"Generating predictions...")
model.eval()
test_preds = []

with torch.no_grad():
    for x, y in test_dataloader:
        x = x.to(device)
        pred = model(x)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds).flatten()

print(f"\n✓ Generated {len(test_preds)} predictions")
print(f"  Range: [{test_preds.min():.2f}°, {test_preds.max():.2f}°]")
print(f"  Mean: {test_preds.mean():.3f}° | Std: {test_preds.std():.3f}°")

# Sanity checks
assert len(test_preds) > 0, "No predictions generated!"
assert not np.isnan(test_preds).any(), "NaN values in predictions!"
assert not np.isinf(test_preds).any(), "Inf values in predictions!"
print("✓ Predictions passed sanity checks")

In [ ]:


test_ground_truths = []
with torch.no_grad():
    for x, y in test_dataloader:
        test_ground_truths.append(y.numpy())
test_ground_truths = np.concatenate(test_ground_truths)

fig = go.Figure()
fig.add_trace(go.Scatter(y=test_ground_truths, mode='lines', name='Placeholder (zeros)', 
                         line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(y=test_preds.flatten(), mode='lines', name='Predictions', 
                         line=dict(color='red', width=1), opacity=0.7))
fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (°)',
    width=CONFIG['PLOTLY_WIDTH'],
    height=CONFIG['PLOTLY_HEIGHT']
)
fig.show()

In [ ]:


y_pred = pd.DataFrame({
    "id": range(len(test_preds)),
    "predicted_sideslip": test_preds
})

y_pred.to_csv(CONFIG['SUBMISSION_CSV'], index=False)

print(f"✅ Predictions saved to: {CONFIG['SUBMISSION_CSV']}")
print(f"   Total samples: {len(test_preds)}")
print(f"   Ready for submission!")

In [ ]:
# ============================================================================
# EVALUATE TRAINING PREDICTIONS WITH CHALLENGE METRIC
# ============================================================================

print("Evaluating training predictions with official challenge metric...\n")

# Create ground truth dataframe
gt_df = pd.DataFrame({"ground_truth_sideslip": ground_truths}, dtype=np.float64)
gt_df["id"] = range(len(ground_truths))
gt_df["band"] = gt_df["ground_truth_sideslip"].round(1).map(shot_band_mapping).fillna("unknown")

# Create predictions dataframe
preds_df = pd.DataFrame({"id": range(len(preds)), "predicted_sideslip": preds})

# Calculate score
score_result = score(gt_df, preds_df, row_id_column_name="id")

print(f"\n{'='*80}")
print(f"CHALLENGE METRIC RESULTS (Training Data)")
print(f"{'='*80}")
print(f"Overall Score: {score_result}")
print(f"\nBand Distribution:")
print(gt_df["band"].value_counts())
print(f"{'='*80}\n")